In [ ]:
# !pip install git+https://github.com/monash-emu/summer3wip.git

In [ ]:
import numpy as np
from plotly import graph_objects as go

import pandas as pd
pd.options.plotting.backend = "plotly"

from summer3.graph import defer, Parameter, CompartmentValues
from summer3.epi import Stratification, CompartmentMap, \
    ManagedArray, CategoryData, TransitionFlow, CompartmentalEpiModel

In [ ]:
population = 1e6  
seed = 1.0
start_time = 0
end_time = 50
model_comps = [
    "susceptible",
    "infectious", 
    "recovered",
]
infect_comps = ["infectious"]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)
epi_model = CompartmentalEpiModel(humans, range(start_time, end_time))

def infection_process(compartment_values: ManagedArray, contact_rate: float, infectious_compartments: tuple):
    infectious_population = compartment_values.query(infectious_compartments).sum()
    total_infectious = compartment_values.sum()
    infectious_prevalence = infectious_population / total_infectious
    return contact_rate * infectious_prevalence

recovery = TransitionFlow("recovery", disease_state["infectious"], disease_state["recovered"], Parameter("recovery_rate", 0.0))
epi_model.add_flow(recovery)
base_parameters = {"effective_contact_rate": 1.2, "recovery_rate": 0.2}

In [ ]:
hosp_strata = ["hosp", "non_hosp"]
hosp_strat = Stratification("hospitalisation", hosp_strata)
humans.stratify(hosp_strat, (disease_state, ["infectious"]))

In [ ]:
vacc_strata = ["vacc", "unvacc"]
vacc_strat = Stratification("vaccination", vacc_strata)
humans.stratify(vacc_strat, (disease_state, ["susceptible"]))

In [ ]:
start_pop = [population - seed, seed, 0.0]
epi_model.set_initial_population(CategoryData(disease_state.categories(), np.array(start_pop)))

In [ ]:
hosp_prop = Parameter("hosp_fraction", 0.0)
for vacc_status in ["vacc", "unvacc"]:
    vacc_effect = Parameter("vacc_effect", 0.0) if vacc_status == "vacc" else 0.0
    for strat in hosp_strata:
        strat_prop = hosp_prop if strat == "hosp" else 1.0 - hosp_prop
        split_force_of_infection = Parameter("effective_contact_rate", 0.0) * strat_prop * (1.0 - vacc_effect)
        force_of_infection = defer(infection_process)(CompartmentValues, split_force_of_infection, disease_state["infectious"])
        infection = TransitionFlow(
            f"infection_{strat}_{vacc_status}", 
            disease_state["susceptible"], 
            (disease_state["infectious"], hosp_strat[strat]), 
            force_of_infection,
        )
        epi_model.add_flow(infection)

In [ ]:
vacc_effects = [0.0, 0.5]
outputs = pd.DataFrame(columns=vacc_effects)
base_parameters = {"effective_contact_rate": 1.2, "recovery_rate": 0.2, "hosp_fraction": 0.5}

for effect in vacc_effects:
    results = epi_model.run(base_parameters | {"vacc_effect": effect})
    outputs[effect] = results["compartments"].to_pandas_df()["infectious_hosp"]

In [ ]:
fig = go.Figure()
legend_format = {"title": "vaccination effect"}
xaxis_format = {"title": "days"}
for c in outputs.columns:
    colour = f"rgba({c}, 0, {1.0 - c}, 1)"
    fig.add_trace(go.Scatter(x=outputs.index, y=outputs[c], mode="lines", name=round(c, 1), line={"color": colour}))
fig.update_layout(title="effect of vaccination on the epidemic", legend=legend_format, xaxis=xaxis_format, yaxis={"title": "infection prevalence"})